[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/editorial-v2/notebooks/11_PID_Control.ipynb)

# Notebook 11 — PID Control

**Companion to Chapter 11**

This notebook adds integral action to reject a constant force disturbance, then exposes the windup created by saturation and tests a back-calculation remedy.

## Learning objectives

By the end of this notebook, you should be able to:

- form the augmented PID closed-loop model;
- check the cubic stability condition;
- compare PD and PID disturbance rejection;
- diagnose windup and evaluate back-calculation anti-windup;

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True})

m = 85.0                 # kg
rho = 1025.0             # kg/m^3
g = 9.80665              # m/s^2
z_star = 20.0            # m, positive downward
V_g0 = 8.0e-3            # m^3 at the surface
p_atm = 101325.0          # Pa
p_star = p_atm + rho*g*z_star
V_g_star = V_g0*p_atm/p_star
k_B = -rho**2*g**2*V_g_star/p_star
a = k_B/m

A = np.array([[0.0, -1.0], [a, 0.0]])
B = np.array([[0.0], [1.0/m]])         # positive input force is upward
C = np.array([[1.0, 0.0]])
D = np.zeros((1, 1))

print(f"Local buoyancy slope k_B = {k_B:.4f} N/m")
print(f"Plant coefficient a = {a:.6f} s^-2")

## 1. PID gains and the augmented poles

With $\eta=\int e\,dt$ and $e=\delta z$,

$$u=K_p e+K_i\eta-K_d\delta v.$$

The characteristic polynomial is

$$s^3+\frac{K_d}{m}s^2+\left(a+\frac{K_p}{m}\right)s+\frac{K_i}{m}.$$

In [ ]:
K_p, K_d, K_i = 150.0, 180.0, 60.0
A_pid = np.array([[0,-1,0], [a+K_p/m,-K_d/m,K_i/m], [1,0,0]])
pid_poles = np.linalg.eigvals(A_pid)
routh_lhs = (K_d/m)*(a+K_p/m)
routh_rhs = K_i/m
print("PID poles:", pid_poles)
print(f"Routh check: {routh_lhs:.3f} > {routh_rhs:.3f}")
assert np.all(pid_poles.real < 0) and routh_lhs > routh_rhs

## 2. PD versus PID under a persistent disturbance

In [ ]:
def simulate_controller(Ki, x0=(0.5,0,0), duration=35, d0=5.0,
                        u_max=np.inf, K_aw=0.0):
    def rhs(t, x):
        z, v, eta = x
        u_raw = K_p*z + Ki*eta - K_d*v
        u = np.clip(u_raw, -u_max, u_max)
        disturbance = d0 if t >= 5 else 0.0
        eta_dot = z + K_aw*(u-u_raw)
        return [-v, a*z+(u+disturbance)/m, eta_dot]
    t_eval = np.linspace(0, duration, int(duration*100)+1)
    sol = solve_ivp(rhs, (0,duration), x0, t_eval=t_eval, rtol=1e-8, atol=1e-10)
    raw = K_p*sol.y[0]+Ki*sol.y[2]-K_d*sol.y[1]
    return sol.t, sol.y, raw, np.clip(raw,-u_max,u_max)

t, x_pd, _, _ = simulate_controller(0)
t, x_pid, _, _ = simulate_controller(K_i)
fig, ax = plt.subplots()
ax.plot(t, x_pd[0], label="PD")
ax.plot(t, x_pid[0], label="PID")
ax.axvline(5, color="k", ls=":", label="disturbance begins")
ax.set(xlabel="Time [s]", ylabel="Depth error [m]", title="Integral action removes constant offset")
ax.legend(); plt.show()
print(f"Final PD error: {x_pd[0,-1]:.4f} m")
print(f"Final PID error: {x_pid[0,-1]:.4f} m")

## 3. Saturation and windup

When the actuator saturates, the integrator can keep accumulating error although its requested force cannot be delivered. This hidden state may then prolong or reverse recovery.

In [ ]:
limit = 35.0
t, x_w, raw_w, app_w = simulate_controller(K_i, x0=(2,0,0), duration=35, d0=0, u_max=limit)
t, x_aw, raw_aw, app_aw = simulate_controller(K_i, x0=(2,0,0), duration=35, d0=0, u_max=limit, K_aw=0.02)

fig, axes = plt.subplots(3,1,sharex=True,figsize=(8,8))
axes[0].plot(t,x_w[0],label="no anti-windup"); axes[0].plot(t,x_aw[0],label="back-calculation")
axes[0].set_ylabel("Depth error [m]"); axes[0].legend()
axes[1].plot(t,x_w[2]); axes[1].plot(t,x_aw[2]); axes[1].set_ylabel("Integral state [m s]")
axes[2].plot(t,app_w,label="applied"); axes[2].plot(t,raw_w,"--",alpha=.6,label="requested")
axes[2].set(xlabel="Time [s]",ylabel="Force [N]"); axes[2].legend()
plt.show()

## 4. Performance metrics

No single metric defines a good controller. Error reduction, overshoot, recovery time, and control effort should be reported together.

In [ ]:
def metrics(t, z, u):
    return {"IAE [m s]": np.trapezoid(np.abs(z),t),
            "peak |error| [m]": np.max(np.abs(z)),
            "RMS force [N]": np.sqrt(np.mean(u**2))}

for label, z, u in [("windup",x_w[0],app_w),("anti-windup",x_aw[0],app_aw)]:
    print(label, metrics(t,z,u))

## Engineering exercises

1. Increase $K_i$ until the Routh condition fails; verify the unstable pole numerically.
2. Sweep the back-calculation gain and compare integral absolute error and RMS force.
3. Add Gaussian velocity-measurement noise. Explain why derivative action raises force variability.
4. Replace back-calculation with conditional integration and compare recovery.


In [ ]:
# Exercise starter: sweep the integral gain and test closed-loop stability
exercise_Ki = np.linspace(0.0, 250.0, 51)
# Store the largest real part of the augmented poles for each gain.


## Summary

Integral action eliminates steady offset from a constant disturbance only when the closed-loop cubic is stable. Saturation makes the controller nonlinear and creates windup; anti-windup is therefore part of the controller, not an optional plotting detail. Chapter 12 evaluates the stable loop across disturbance frequencies.